# 7장. RAG 시스템 평가 — 08. 온라인 평가 (Online Evaluation)

## 학습 목표

- **Offline 평가**와 **Online 평가(온라인 평가)**의 차이와 각각의 필요성 이해
- LangSmith Tracing으로 프로덕션 요청을 자동 로깅
- **RunnableParallel**로 온라인 평가용 구조화 출력(context, answer, question) 구성
- LangSmith UI에서 **Online LLM-as-Judge** 평가자 설정 방법 이해
- **Tag 기반 선택적 평가**로 비용 효율적인 프로덕션 평가 운영
- Latency(지연 시간), Token 사용량, Error Rate 등 실시간 모니터링 지표 이해

## 사전 지식

- 03~07번 노트북에서 Offline 평가 경험
- LangSmith 환경 설정

## Offline vs Online 평가 비교

```mermaid
flowchart LR
    subgraph Offline["Offline 평가 (개발 단계)"]
        direction TB
        O1[테스트 데이터셋<br/>준비]
        O2[evaluate 실행]
        O3[지표 점수 확인]
        O4[모델/파라미터<br/>개선]
        O1 --> O2 --> O3 --> O4 --> O1
    end

    subgraph Online["Online 평가 (프로덕션)"]
        direction TB
        N1[실제 사용자<br/>요청]
        N2[자동 Tracing<br/>LangSmith]
        N3[Online Evaluator<br/>LLM-as-Judge]
        N4[이상 감지<br/>및 알림]
        N1 --> N2 --> N3 --> N4
    end

    Offline -->|배포| Online

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460

    class O1,N1 input
    class O2,O3,N2,N3 process
    class O4,N4 output
```

두 가지 평가 방식은 **상호 보완적**이에요. Offline으로 검증한 후 배포하고, Online으로 실제 운영 품질을 모니터링합니다.

**핵심 차이**: Offline 평가는 `evaluate()` 함수로 코드에서 실행하지만, Online 평가는 **LangSmith UI에서 설정**하면 트레이스가 생성될 때 **자동으로 평가가 실행**됩니다.

> 🎯 **강의 포인트**: Offline 평가는 배포 전 검증이고, Online 평가는 배포 후 모니터링입니다. 두 가지 모두 필요합니다. Offline으로 통과한 시스템도 실제 사용자 쿼리에서는 예상치 못한 실패가 발생할 수 있기 때문에, Online 모니터링은 선택이 아닌 필수입니다.

> 🔑 **핵심 개념**: `LANGCHAIN_TRACING_V2=true` 환경 변수 하나로 **코드를 전혀 수정하지 않고** 모든 LangChain 실행이 자동으로 LangSmith에 기록됩니다. 프로덕션 배포 시 이 환경 변수만 추가하면 됩니다. 이것이 LangSmith의 가장 강력한 기능입니다.

---

## LangSmith Tracing 활성화

`LANGCHAIN_TRACING_V2=true`로 설정하면 이후 모든 LangChain 체인 실행이 자동으로 LangSmith에 기록돼요. 코드를 전혀 수정하지 않아도 됩니다.

> ⚠️ **자주 하는 실수**: Tracing을 켜면 모든 요청이 LangSmith에 전송되어 **추가 비용과 지연**이 발생합니다. 프로덕션에서는 100% 추적 대신 샘플링(예: 10% 요청만 추적)을 고려하세요. `LANGCHAIN_TRACING_SAMPLING_RATE` 환경 변수로 비율을 조정할 수 있습니다.

In [ ]:
# ---------------------------------------------------
# LangSmith Tracing 활성화 — 이후 모든 체인 실행이 자동 로깅됨
# ---------------------------------------------------
from dotenv import load_dotenv
import os

load_dotenv()

# macOS에서 FAISS 사용 시 OpenMP 중복 로드 방지
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# LANGSMITH_PROJECT: LangSmith 대시보드에서 구분할 프로젝트 이름
# (.env의 LANGSMITH_TRACING=true 설정으로 자동 추적이 활성화됩니다)
os.environ["LANGSMITH_PROJECT"] = "08-Eval-Online-Monitoring"

print("LangSmith Tracing 활성화")
print(f"Project: {os.getenv('LANGSMITH_PROJECT')}")

# ---------------------------------------------------
# [LangSmith UI 확인 방법]
# 1. https://smith.langchain.com 접속
# 2. 좌측 Projects 클릭
# 3. "08-Eval-Online-Monitoring" 프로젝트 선택
# 4. 주안점: Traces 목록 → 실시간 추적 확인, Rules 탭 → 온라인 평가 규칙
# ---------------------------------------------------

---

## RAG 시스템 (자동 로깅 + 구조화 출력)

Tracing이 활성화된 상태에서 구축한 RAG 시스템은 모든 요청이 자동으로 LangSmith에 기록돼요. 별도의 평가 코드 없이도 트레이스(trace)가 남습니다.

**중요**: Online 평가자가 `context`, `answer`, `question`을 각각 참조할 수 있도록 `RunnableParallel`로 **구조화된 출력**을 구성합니다. 단순히 answer 문자열만 반환하면 온라인 평가자가 context를 확인할 수 없어요.

> 💡 **실무 팁**: 프로덕션 배포 시 환경 변수 `LANGCHAIN_TRACING_V2=true`만 설정하면 모든 LangChain 기반 코드의 실행이 자동으로 추적됩니다. 비용이 발생하므로 샘플링 비율은 필요에 따라 조정하세요.

> 💡 **실무 팁**: LangSmith 대시보드에서 주목해야 할 핵심 지표 3가지 — **Latency**(P95 기준 느린 요청 탐지), **Token Usage**(비용 급증 감지), **Error Rate**(시스템 장애 조기 발견). 이 세 가지에 알림을 설정하면 문제가 사용자에게 영향을 미치기 전에 대응할 수 있습니다.

In [2]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ---------------------------------------------------
# 트랜스포머 관련 한국어 컨텍스트 문서
# ---------------------------------------------------
documents = [
    Document(page_content=(
        "트랜스포머(Transformer)는 2017년 구글이 'Attention Is All You Need' 논문에서 "
        "제안한 신경망 아키텍처입니다. 기존의 순환 신경망(RNN)이나 합성곱 신경망(CNN) 없이 "
        "어텐션 메커니즘만으로 시퀀스를 처리합니다. 인코더와 디코더 각각 6개의 동일한 레이어를 "
        "쌓아 올린 구조이며, 모든 레이어의 출력 차원은 d_model = 512입니다."
    )),
    Document(page_content=(
        "셀프 어텐션(Self-Attention)은 시퀀스 내 모든 위치 간의 관계를 계산하는 메커니즘입니다. "
        "입력에서 쿼리(Query), 키(Key), 값(Value) 벡터를 생성하고, 쿼리와 키의 유사도로 "
        "어텐션 가중치를 계산합니다. 이를 통해 모델이 입력의 어떤 부분에 집중해야 하는지 학습합니다."
    )),
    Document(page_content=(
        "멀티헤드 어텐션(Multi-Head Attention)은 셀프 어텐션을 여러 개의 '헤드'로 병렬 수행하는 "
        "기법입니다. 각 헤드는 서로 다른 표현 부분공간(representation subspace)에서 정보를 추출합니다. "
        "논문에서는 8개의 헤드를 사용했으며, 각 헤드의 출력을 연결한 뒤 선형 변환을 적용합니다."
    )),
    Document(page_content=(
        "포지셔널 인코딩(Positional Encoding)은 트랜스포머에 단어의 위치 정보를 제공합니다. "
        "트랜스포머는 순환 구조가 없어 단어의 순서를 알 수 없으므로, 사인(sin)과 코사인(cos) "
        "함수 기반의 위치 벡터를 입력 임베딩에 더해줍니다."
    )),
    Document(page_content=(
        "트랜스포머의 인코더는 입력 시퀀스를 연속적인 표현으로 변환하고, 디코더는 이를 바탕으로 "
        "출력을 한 토큰씩 생성합니다. 디코더에는 마스크드 셀프 어텐션이 추가되어 미래 토큰 참조를 "
        "방지합니다. 인코더-디코더 어텐션으로 디코더가 입력의 관련 부분에 집중합니다."
    )),
    Document(page_content=(
        "트랜스포머는 RNN과 달리 시퀀스를 병렬로 처리할 수 있어 훈련 속도가 크게 빠릅니다. "
        "RNN은 순차 처리가 필수지만 트랜스포머는 모든 위치를 동시에 처리합니다. "
        "장거리 의존성도 더 효과적으로 포착할 수 있습니다."
    )),
    Document(page_content=(
        "스케일드 닷 프로덕트 어텐션은 쿼리와 키의 내적을 키 차원의 제곱근으로 나누어 스케일링합니다. "
        "스케일링 없이는 내적 값이 커져 소프트맥스의 기울기가 매우 작아집니다. "
        "스케일링 후 소프트맥스를 적용하여 가중치를 구하고, 값 벡터에 곱하여 출력을 얻습니다."
    )),
]

# 벡터 DB 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever()

# 3단계: 프롬프트 정의
prompt = PromptTemplate.from_template(
    """주어진 컨텍스트를 바탕으로 답변하세요.

컨텍스트: {context}
질문: {question}
답변:"""
)

# 4단계: LLM 및 체인 생성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 내부 RAG 체인: context + question -> 답변 생성
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5단계: Online 평가용 구조화 출력
# RunnableParallel로 context, answer, question을 모두 output에 포함시킴
# -> LangSmith Online Evaluator가 output.context, output.answer 등을 참조 가능
evaluation_chain = RunnableParallel(
    {
        "context": retriever,       # 검색된 문서 (평가자가 근거 확인용)
        "answer": rag_chain,        # RAG 답변
        "question": RunnablePassthrough(),  # 원본 질문
    }
)

print("RAG 시스템 구축 완료 (LangSmith가 자동 로깅)")
print("  - rag_chain: 답변만 반환 (일반 사용)")
print("  - evaluation_chain: context + answer + question 반환 (온라인 평가용)")

RAG 시스템 구축 완료 (LangSmith가 자동 로깅)
  - rag_chain: 답변만 반환 (일반 사용)
  - evaluation_chain: context + answer + question 반환 (온라인 평가용)


> 🔑 **핵심 개념**: `RunnableParallel`로 구조화된 출력을 만드는 이유는 **Online 평가자가 output의 개별 필드를 참조**하기 때문이에요. 예를 들어 Hallucination 평가자는 `output.context`(근거)와 `output.answer`(답변)를 비교해야 합니다. 답변 문자열만 반환하는 체인에서는 이런 평가가 불가능합니다.

```
# LangSmith에 기록되는 트레이스 구조:
{
  "input": "트랜스포머 아키텍처란 무엇인가요?",
  "output": {
    "context": [Document(...), Document(...)],  # <- 평가자가 참조
    "answer": "트랜스포머는...",           # <- 평가자가 참조
    "question": "트랜스포머 아키텍처란..."     # <- 평가자가 참조
  }
}
```

---

## 시뮬레이션: 프로덕션 트래픽

실제 사용자 요청을 시뮬레이션해서 LangSmith에 트레이스를 남겨볼게요. `evaluation_chain`을 사용하면 각 요청의 context, answer, question이 모두 LangSmith 트레이스에 기록됩니다. 이후 대시보드에서 각 요청의 Latency, Token 사용량, 전체 실행 과정을 확인할 수 있어요.

In [ ]:
# ---------------------------------------------------
# 프로덕션 트래픽 시뮬레이션 — 실행 결과가 LangSmith에 자동 기록됨
# ---------------------------------------------------

# 시뮬레이션: 여러 사용자 요청
user_questions = [
    "트랜스포머 아키텍처란 무엇인가요?",
    "셀프 어텐션은 어떻게 작동하나요?",
    "트랜스포머의 장점은 무엇인가요?",
    "멀티헤드 어텐션을 설명해주세요.",
    "포지셔널 인코딩의 역할은 무엇인가요?",
    "인코더-디코더 구조란 무엇인가요?",
    "스케일드 닷 프로덕트 어텐션이란 무엇인가요?",
]

print(f"시뮬레이션 시작: {len(user_questions)}개의 사용자 요청 처리\n")

for i, question in enumerate(user_questions, 1):
    print(f"[요청 {i}] {question}")
    # evaluation_chain: context + answer + question이 모두 output에 포함됨
    result = evaluation_chain.invoke(question)
    print(f"[응답] {result['answer'][:100]}...\n")

print("시뮬레이션 완료!")
print("\nLangSmith 대시보드에서 확인:")
print("  - Latency: 각 요청의 응답 시간")
print("  - Token Usage: 토큰 사용량 및 비용")
print("  - Traces: 각 요청의 상세 실행 로그")
print(f"\nhttps://smith.langchain.com 에서 프로젝트 '{os.getenv('LANGSMITH_PROJECT')}' 확인")

---

## Online LLM-as-Judge 설정 (LangSmith UI)

트레이스가 LangSmith에 기록되면, **코드 없이 UI에서** Online 평가자(LLM-as-Judge)를 설정할 수 있어요. 설정한 평가자는 새로운 트레이스가 생성될 때마다 자동으로 실행됩니다.

```mermaid
flowchart LR
    A[사용자 요청] --> B[체인 실행]
    B --> C[LangSmith 트레이스 기록]
    C --> D[Online Evaluator 자동 실행]
    D --> E[평가 점수 대시보드에 표시]

    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#d4edda,stroke:#28a745,color:#155724
    class A,B,C process
    class D,E output
```

### 설정 방법 (Step-by-Step)

아래 단계를 LangSmith 웹 UI에서 진행합니다.

**Step 1: Online Evaluator 생성**
1. [smith.langchain.com](https://smith.langchain.com) 접속 후 해당 프로젝트로 이동
2. 프로젝트 페이지 상단의 **"Online Evaluation"** 탭 클릭
3. **"+ Add online evaluator"** 버튼 클릭
4. **"LLM-as-judge"** 유형 선택

**Step 2: API Key 및 모델 설정**
1. **Secrets & API Keys** 섹션에서 OpenAI API Key를 입력
2. **Provider**: `OpenAI` 선택
3. **Model**: `gpt-4o-mini` 선택 (비용 효율적)
4. **Prompt**: 평가 기준에 맞는 프롬프트 작성 (예: Hallucination 탐지, 답변 품질 등)

**Step 3: 입력 변수 매핑**
- Online 평가자의 프롬프트에서 사용할 변수를 트레이스 데이터에 매핑합니다.
- 예시 (Hallucination 평가 기준):
  - `facts` (근거) -> `output.context` 에 매핑
  - `answer` (답변) -> `output.answer` 에 매핑
  - `question` (질문) -> `output.question` 에 매핑

**Step 4: Preview 확인**
- **Preview** 버튼을 눌러 실제 트레이스 데이터가 올바르게 매핑되었는지 확인
- 데이터가 정상적으로 표시되면 Preview 모드를 **반드시 꺼야** 다음 단계로 진행 가능

**Step 5: 저장 및 활성화**
- Grade 결과를 확인하고 **Save** 버튼으로 저장
- 이후 새로운 트레이스가 생성될 때마다 자동으로 평가가 실행됨

> 🎯 **강의 포인트**: Online Evaluator는 **코드 변경 없이** LangSmith UI에서만 설정합니다. 프로덕션 코드를 수정하지 않고도 평가 기준을 추가하거나 변경할 수 있어요. 이것이 Offline 평가와의 핵심 차이점입니다.

> ⚠️ **자주 하는 실수**: Online Evaluator의 입력 변수 매핑에서 `output.context`를 지정하려면 반드시 `RunnableParallel`로 구조화된 출력을 사용해야 합니다. 단순히 `StrOutputParser()`로 문자열만 반환하면 `output`이 문자열이므로 `output.context` 같은 필드 접근이 불가능해요.

---

## Tag 기반 선택적 평가

모든 트레이스에 Online Evaluator를 실행하면 LLM 호출 비용이 급증해요. **Tag를 설정하면 특정 태그가 붙은 트레이스에만 평가를 실행**할 수 있습니다.

### LangSmith UI에서 Tag 필터 설정 방법

1. 프로젝트의 Online Evaluation 탭에서 기존 평가자의 **Edit Rule** 클릭
2. **Filter by tags** 섹션에서 원하는 태그 이름 입력 (예: `hallucination_eval`)
3. 저장하면 해당 태그가 포함된 트레이스에서만 평가자가 실행됨

### 코드에서 Tag 추가하기

`RunnableConfig(tags=[...])` 를 사용해서 체인 실행 시 태그를 부여합니다.

In [4]:
# ---------------------------------------------------
# Tag 기반 선택적 평가 — RunnableConfig로 태그 부여
# ---------------------------------------------------
from langchain_core.runnables import RunnableConfig

# 평가 목적별 Config 정의
# LangSmith UI에서 각 태그에 매칭되는 Online Evaluator를 설정
hallucination_config = RunnableConfig(tags=["hallucination_eval"])
context_recall_config = RunnableConfig(tags=["context_recall_eval"])
# 여러 태그를 동시에 부여하면 해당 태그를 필터로 사용하는 모든 평가자가 실행됨
all_eval_config = RunnableConfig(tags=["hallucination_eval", "context_recall_eval"])

print("Tag 기반 Config 정의 완료:")
print(f"  - hallucination_config: {hallucination_config.get('tags')}")
print(f"  - context_recall_config: {context_recall_config.get('tags')}")
print(f"  - all_eval_config: {all_eval_config.get('tags')}")

Tag 기반 Config 정의 완료:
  - hallucination_config: ['hallucination_eval']
  - context_recall_config: ['context_recall_eval']
  - all_eval_config: ['hallucination_eval', 'context_recall_eval']


In [5]:
# ---------------------------------------------------
# Tag별 선택적 평가 실행 예시
# ---------------------------------------------------
test_question = "트랜스포머에서 포지셔널 인코딩의 역할은 무엇인가요?"

# 1) 태그 없이 실행 -> Online Evaluator가 실행되지 않음 (또는 전체 실행 설정 시 모두 실행)
print("[1] 태그 없이 실행 (평가 없음)")
result = evaluation_chain.invoke(test_question)
print(f"    답변: {result['answer'][:80]}...\n")

# 2) Hallucination 평가만 실행
print("[2] hallucination_eval 태그 -> Hallucination 평가만 트리거")
result = evaluation_chain.invoke(test_question, config=hallucination_config)
print(f"    답변: {result['answer'][:80]}...\n")

# 3) Context Recall 평가만 실행
print("[3] context_recall_eval 태그 -> Context Recall 평가만 트리거")
result = evaluation_chain.invoke(test_question, config=context_recall_config)
print(f"    답변: {result['answer'][:80]}...\n")

# 4) 모든 평가 실행
print("[4] 모든 태그 -> Hallucination + Context Recall 평가 모두 트리거")
result = evaluation_chain.invoke(test_question, config=all_eval_config)
print(f"    답변: {result['answer'][:80]}...\n")

print("Tag 기반 선택적 평가 시뮬레이션 완료!")
print("LangSmith 대시보드에서 각 트레이스의 태그와 평가 결과를 확인하세요.")

[1] 태그 없이 실행 (평가 없음)


    답변: The role of positional encoding in the Transformer is to provide information abo...

[2] hallucination_eval 태그 -> Hallucination 평가만 트리거


    답변: The role of positional encoding in the Transformer is to provide information abo...

[3] context_recall_eval 태그 -> Context Recall 평가만 트리거


    답변: The role of positional encoding in the Transformer is to provide information abo...

[4] 모든 태그 -> Hallucination + Context Recall 평가 모두 트리거


    답변: The role of positional encoding in the Transformer is to provide information abo...

Tag 기반 선택적 평가 시뮬레이션 완료!
LangSmith 대시보드에서 각 트레이스의 태그와 평가 결과를 확인하세요.


---

## 트레이싱 결과 해석

Online 평가는 `evaluate()`가 아닌 **LangSmith 대시보드에서 직접 확인**합니다. 아래는 대시보드에서 확인할 수 있는 주요 지표와 해석 방법입니다.

### 트레이스 목록에서 확인할 항목

| 항목 | 설명 | 주의할 점 |
|------|------|----------|
| **Latency** | 각 요청의 전체 응답 시간 | 3초 이상이면 사용자 경험에 영향 |
| **Token Usage** | 입력 + 출력 토큰 수 | 급증하면 프롬프트나 컨텍스트 길이 확인 |
| **Status** | 성공(200) / 실패(에러) | 에러 발생 시 Run Tree에서 실패 단계 확인 |
| **Tags** | 코드에서 부여한 태그 | 태그별 필터링으로 선택적 분석 가능 |

### 프로젝트 대시보드 집계 지표

| 지표 | 의미 | 실무 활용 |
|------|------|----------|
| **P50 Latency** | 중간값 응답 시간 — 일반적인 속도 | 기준선(baseline) 설정에 사용 |
| **P99 Latency** | 99번째 백분위 응답 시간 — 최악의 경우 | P50과 차이가 크면 일부 요청에서 병목 발생 |
| **Error Rate** | 전체 요청 중 오류 비율 | 0%가 아니면 즉시 확인 필요 |
| **Run Count** | 추적된 총 요청 수 | 트래픽 규모 파악 |

> 💡 **실무 팁**: P50이 2초인데 P99가 10초라면, 대부분의 요청은 빠르지만 일부가 극단적으로 느린 것입니다. P99 요청의 Run Tree를 열어 어느 단계(검색? LLM 호출?)에서 시간이 걸리는지 확인하세요.

---

## 7장 전체 마무리

### RAG 시스템 평가 전체 흐름 요약

7장에서 배운 모든 평가 방법을 한눈에 정리해볼게요.

```mermaid
flowchart TD
    A[RAG 시스템 구축<br/>6장] --> B[01. 테스트 데이터셋 생성<br/>RAGAS 합성 데이터]
    B --> C[02. RAGAS 평가<br/>4가지 지표]
    C --> D[03. LangSmith<br/>데이터셋 + LLM-as-Judge]
    D --> E[04. 커스텀 평가자<br/>규칙 / LLM / 임베딩]
    E --> F[05. Heuristic 평가<br/>ROUGE / BLEU / METEOR / SemScore]
    F --> G[06. Groundedness 평가<br/>할루시네이션 탐지]
    G --> H[07. 모델 비교<br/>A/B 테스트]
    H --> I[08. Online 평가<br/>실시간 모니터링]
    I --> J{품질 목표<br/>달성?}
    J -->|아니오| K[개선: 임베딩 / 청크 /<br/>프롬프트 / 모델]
    K --> C
    J -->|예| L[프로덕션 배포]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24

    class A input
    class B,C,D,E,F,G,H,I process
    class J,K error
    class L output
```

### 평가 방법 선택 가이드

| 상황 | 추천 방법 |
|------|---------|
| RAG 파이프라인 첫 평가 | RAGAS 4가지 지표 |
| 빠른 반복 개발 중 | 규칙 기반 커스텀 평가자 |
| 참조 답변이 있을 때 | ROUGE + SemScore (Heuristic) |
| 할루시네이션 우려 | Groundedness 평가 |
| 모델 교체 검토 | LangSmith 모델 비교 |
| 프로덕션 운영 중 | Online Tracing + 모니터링 |
| 종합 품질 보증 | 위 방법 조합 |

### 7장에서 배운 핵심 개념 정리

- **RAGAS**: RAG 특화 자동 평가 프레임워크 (합성 데이터 생성 + 4가지 지표)
- **LangSmith**: 클라우드 기반 평가 플랫폼 (데이터셋 관리 + 실험 추적 + 모니터링)
- **LLM-as-Judge**: LLM을 평가자로 활용하는 자동화 패턴
- **Heuristic 평가**: LLM 없이 통계적으로 품질 측정 (ROUGE, BLEU, METEOR, SemScore)
- **Groundedness**: 할루시네이션 탐지를 위한 근거성 평가
- **Offline vs Online**: 개발 단계 평가 vs 프로덕션 모니터링의 상호 보완적 관계

좋은 RAG 시스템은 한 번의 평가로 완성되지 않아요. 지속적인 평가와 개선의 순환이 품질을 높여갑니다.